# Caseware Customer Voice — Full NLP Pipeline

**Data Parsing → VADER Sentiment → Pain Point Categorization → TF-IDF + NMF Topic Modeling**

Built by Hammad Mirza | April 2026

---

This notebook walks through the complete pipeline for analyzing 221 customer reviews of Caseware products from G2, Capterra, Software Advice, and Reddit.

## Step 0: Install & Import

In [ ]:
!pip install vaderSentiment scikit-learn pandas plotly wordcloud matplotlib -q

In [ ]:
import pandas as pd
import numpy as np
import re
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import NMF
from wordcloud import WordCloud, STOPWORDS as WC_STOPWORDS
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt

print('All imports loaded.')

---
## Step 1: Load & Clean Data

The raw text files from G2, Capterra, Software Advice, and Reddit were pre-parsed using `parse_reviews.py` into a structured CSV. Each row has: `source`, `rating`, `text`, `pros`, `cons`.

**Reddit filtering:** Raw Reddit threads were filtered from 279 comments to 94 by keeping only comments containing Caseware-related keywords (caseware, working papers, DAS, trial balance, cloud version, etc.).

In [ ]:
# Upload reviews.csv to Colab first, or mount Google Drive
df = pd.read_csv('reviews.csv')

df['text'] = df['text'].fillna('').astype(str)
df['pros'] = df['pros'].fillna('').astype(str)
df['cons'] = df['cons'].fillna('').astype(str)
df = df[df['text'].str.len() > 20].reset_index(drop=True)

print(f'Total reviews: {len(df)}')
print(f'\nBy source:')
print(df['source'].value_counts())
print(f'\nReviews with star ratings: {len(df[df.rating > 0])}')
print(f'Average star rating: {df[df.rating > 0]["rating"].mean():.2f}')

In [ ]:
# Preview one review from each source
for source in df['source'].unique():
    sample = df[df['source'] == source].iloc[0]
    print(f'--- {source} ---')
    print(f'Rating: {sample["rating"]}')
    print(f'Text: {sample["text"][:150]}...')
    print()

---
## Step 2: VADER Sentiment Analysis

**What VADER does:** A lexicon-based tool with ~7,500 pre-scored words. For each review it:
1. Looks up every word in its lexicon (e.g. "excellent" = +3.2, "terrible" = -3.1)
2. Applies rules for negation ("not good"), intensifiers ("very good"), capitalization, punctuation
3. Returns a compound score from -1 (most negative) to +1 (most positive)

**Classification thresholds:** compound >= 0.05 = Positive, <= -0.05 = Negative, else Neutral.

In [ ]:
# Single review example — see how VADER scores it
analyzer = SentimentIntensityAnalyzer()

sample = df['text'].iloc[0]
scores = analyzer.polarity_scores(sample)

print(f'Text: {sample[:150]}...')
print(f'\nPositive proportion: {scores["pos"]:.3f}')
print(f'Neutral proportion:  {scores["neu"]:.3f}')
print(f'Negative proportion: {scores["neg"]:.3f}')
print(f'Compound score:      {scores["compound"]:.3f}')

In [ ]:
# Score every review
sentiments = df['text'].apply(lambda x: analyzer.polarity_scores(x))
df['vader_compound'] = sentiments.apply(lambda x: x['compound'])
df['sentiment'] = df['vader_compound'].apply(
    lambda x: 'Positive' if x >= 0.05 else ('Negative' if x <= -0.05 else 'Neutral')
)

print('Sentiment distribution:')
print(df['sentiment'].value_counts())
print(f'\nMean compound score: {df["vader_compound"].mean():.3f}')

In [ ]:
# Sentiment by source
print('Mean sentiment by source:')
print(df.groupby('source')['vader_compound'].agg(['mean', 'count']).round(3).sort_values('mean'))
print('\nReddit has the lowest sentiment (unfiltered practitioner voice).')
print('G2 has the highest (reviews are often incentivized).')

In [ ]:
# Visualize: sentiment by source
source_sent = df.groupby('source')['vader_compound'].mean().sort_values()
colors = ['#e74c3c' if v < -0.05 else '#27ae60' if v > 0.05 else '#f39c12' for v in source_sent.values]

fig = go.Figure(go.Bar(
    x=source_sent.values, y=source_sent.index, orientation='h',
    marker_color=colors, text=[f'{v:.2f}' for v in source_sent.values], textposition='auto'
))
fig.update_layout(title='Mean sentiment by source', xaxis=dict(range=[-1, 1]), height=300)
fig.show()

In [ ]:
# Most positive and most negative reviews
print('TOP 3 MOST POSITIVE:')
for _, row in df.nlargest(3, 'vader_compound').iterrows():
    print(f'  [{row["source"]}] {row["vader_compound"]:.2f} | {row["text"][:100]}...')

print('\nTOP 3 MOST NEGATIVE:')
for _, row in df.nsmallest(3, 'vader_compound').iterrows():
    print(f'  [{row["source"]}] {row["vader_compound"]:.2f} | {row["text"][:100]}...')

---
## Step 3: Word Clouds

Visualize the most frequent words in positive vs. negative reviews. Stop words are removed to surface meaningful terms only.

In [ ]:
# Shared stop words for word clouds and topic modeling
CUSTOM_STOPS = set(ENGLISH_STOP_WORDS) | {
    'caseware', 'software', 'product', 'tool', 'program', 'application', 'app',
    'use', 'using', 'used', 'user', 'users', 'review', 'reviews', 'reviewer',
    'recommend', 'recommended', 'overall', 'experience', 'rating', 'ratings',
    'good', 'great', 'best', 'better', 'worst', 'nice', 'excellent', 'amazing',
    'like', 'really', 'well', 'just', 'much', 'lot', 'able', 'also',
    'would', 'could', 'should', 'might', 'may', 'shall', 'will', 'can',
    'make', 'made', 'makes', 'get', 'got', 'gets', 'know', 'known',
    'think', 'want', 'need', 'needs', 'way', 'ways', 'thing', 'things',
    'time', 'times', 'year', 'years', 'month', 'months', 'week', 'day',
    'work', 'working', 'works', 'worked', 'come', 'comes', 'came',
    'take', 'takes', 'took', 'find', 'found', 'give', 'gives', 'gave',
    'tell', 'look', 'looks', 'try', 'tried', 'trying',
    'new', 'old', 'first', 'last', 'right', 'sure', 'even', 'still',
    'though', 'however', 'although', 'said', 'say', 'says', 'see', 'seen',
    'going', 'go', 'goes', 'went', 'done', 'doing', 'did', 'does',
    'put', 'set', 'let', 'run', 'end', 'far', 'bit', 'big', 'bad',
    'back', 'long', 'keep', 'kept', 'start', 'started', 'point',
    'actually', 'pretty', 'quite', 'enough',
    # Scraping artifacts
    'collected', 'hosted', 'com', 'thumbnail', 'sidebar', 'promoted',
    'post', 'image', 'avatar', 'show', 'details', 'read', 'click',
    'sign', 'join', 'log', 'continue', 'skip', 'content', 'navigation',
    'reply', 'upvote', 'downvote', 'share', 'comments', 'comment',
    'ago', 'deleted', 'removed', 'reddit', 'subreddit',
    'source', 'verified', 'linkedin', 'profile',
    # Platform boilerplate
    'validated', 'incentivized', 'invite', 'emp', 'guest', 'upvotes',
    'conversation', 'community', 'ask', 'asked',
    'answers', 'real', 'honest', 'pros', 'cons', 'website', 'learn',
    'information', 'comparisons', 'practical', 'recommendations',
    'alternatives', 'view', 'explore', 'workflows', 'discussions',
    'currently', 'available', 'visit', 'fewer',
    'less', 'more', 'reviewed', 'employees', 'management',
    'none', 'full', 'helpful', 'pricing',
    # Foreign fragments
    'el', 'em', 'en', 'fr', 'je', 'la', 'las', 'los', 'que', 'se',
    'uso', 'uma', 'da', 'es', 'muy', 'por', 'para', 'mas', 'nos',
    'con', 'del', 'les', 'des', 'une',
    # Short noise
    'gu', 'orb', 'vpa', 'vr', 'bin', 'lol', 'hey', 'ok', 'yes',
    'guy', 'guys', 'fan', 'pay', 'job', 'tr', 'ic',
    'g2', 'capterra',
}

print(f'Total stop words: {len(CUSTOM_STOPS)}')

In [ ]:
# Generate word clouds
wc_stops = set(WC_STOPWORDS) | CUSTOM_STOPS

pos_text = ' '.join(df[df['sentiment'] == 'Positive']['text'].tolist())
neg_text = ' '.join(
    df[df['sentiment'] == 'Negative']['text'].tolist() +
    df[df['cons'].str.len() > 5]['cons'].tolist()
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

wc_pos = WordCloud(width=600, height=300, background_color='white',
                   colormap='Greens', max_words=60, stopwords=wc_stops,
                   collocations=False, min_word_length=3).generate(pos_text)
ax1.imshow(wc_pos, interpolation='bilinear')
ax1.set_title('Positive reviews', fontsize=14)
ax1.axis('off')

wc_neg = WordCloud(width=600, height=300, background_color='white',
                   colormap='Reds', max_words=60, stopwords=wc_stops,
                   collocations=False, min_word_length=3).generate(neg_text)
ax2.imshow(wc_neg, interpolation='bilinear')
ax2.set_title('Negative reviews & cons', fontsize=14)
ax2.axis('off')

plt.tight_layout()
plt.show()

print('Key contrast: Positive reviews emphasize "easy", "efficient", "engagement".')
print('Negative reviews emphasize "training", "learn", "interface", "slow", "expensive".')

---
## Step 4: Pain Point Categorization

Takes every negative review and every cons section, then matches against keyword lists for 8 categories. A single review can match multiple categories.

**Why keyword-based instead of ML:** For pain points, predefined categories like "Learning Curve" or "Cloud & Migration" are directly actionable for a PS team. The categories were designed based on manual review of the data.

In [ ]:
# Define pain point categories
pain_categories = {
    'User Interface & Usability': [
        'interface', 'clunky', 'unintuitive', 'confusing', 'navigate',
        'navigation', 'cluttered', 'dated', 'modern', 'layout',
        'user friendly', 'cumbersome', 'complicated', 'complex',
        'overwhelming', 'intuitive', 'clumsy', 'awkward', 'frustrating'
    ],
    'Learning Curve & Training': [
        'learning curve', 'steep', 'training', 'difficult to learn',
        'onboarding', 'documentation', 'tutorial', 'manual',
        'figure out', 'getting started', 'hard to understand'
    ],
    'Performance & Speed': [
        'slow', 'lag', 'loading', 'freeze', 'crash', 'hang', 'speed',
        'performance', 'unresponsive', 'timeout', 'sluggish'
    ],
    'Cloud & Migration': [
        'cloud', 'migration', 'migrate', 'desktop', 'transition', 'upgrade',
        'version', 'cloudbridge', 'cloud version', 'desktop version'
    ],
    'Cost & Licensing': [
        'expensive', 'cost', 'price', 'license', 'licensing',
        'subscription', 'fee', 'money', 'overpriced'
    ],
    'Customer Support': [
        'support', 'customer service', 'help desk', 'ticket',
        'response time', 'technical support'
    ],
    'Features & Functionality': [
        'feature', 'functionality', 'missing', 'lack', 'limited',
        'customization', 'customize', 'flexibility', 'integration',
        'export', 'import', 'template', 'formatting'
    ],
    'Updates & Bugs': [
        'bug', 'error', 'glitch', 'fix', 'update', 'patch',
        'broken', 'not working', 'fails', 'problem', 'malfunction'
    ]
}

print(f'{len(pain_categories)} categories defined')

In [ ]:
# Collect complaint texts and match against categories
negative_reviews = df[df['sentiment'] == 'Negative']['text'].tolist()
cons_texts = df[df['cons'].str.len() > 5]['cons'].tolist()
all_complaints = negative_reviews + cons_texts

print(f'Negative reviews: {len(negative_reviews)}')
print(f'Cons sections: {len(cons_texts)}')
print(f'Total complaint texts: {len(all_complaints)}')

pain_results = {}
for category, keywords in pain_categories.items():
    matches = [t for t in all_complaints if any(kw in t.lower() for kw in keywords)]
    pain_results[category] = {
        'count': len(matches),
        'pct': round(len(matches) / max(len(all_complaints), 1) * 100, 1),
        'samples': matches[:2]
    }

pain_results = dict(sorted(pain_results.items(), key=lambda x: x[1]['count'], reverse=True))

print('\nPain point ranking:')
for cat, data in pain_results.items():
    print(f'  {cat}: {data["count"]} mentions ({data["pct"]}%)')

In [ ]:
# Visualize pain points
pain_df = pd.DataFrame([
    {'Category': cat, 'Mentions': data['count']}
    for cat, data in pain_results.items() if data['count'] > 0
]).sort_values('Mentions', ascending=True)

fig = go.Figure(go.Bar(
    x=pain_df['Mentions'], y=pain_df['Category'], orientation='h',
    marker_color='#e74c3c', text=pain_df['Mentions'], textposition='auto'
))
fig.update_layout(title='Pain point frequency', height=400)
fig.show()

In [ ]:
# Sample quotes for top 3 pain points
for cat, data in list(pain_results.items())[:3]:
    print(f'--- {cat} ({data["count"]} mentions) ---')
    for sample in data['samples']:
        print(f'  "{sample[:150]}..."')
    print()

---
## Step 5: Topic Modeling (TF-IDF + NMF)

### Step 5a: TF-IDF (Term Frequency-Inverse Document Frequency)
Converts text into numerical vectors. For each word in each review:
- **TF**: How often does this word appear in THIS review?
- **IDF**: How rare is this word across ALL reviews?
- **TF-IDF = TF x IDF**: Words frequent in one review but rare overall get high scores.

### Step 5b: NMF (Non-negative Matrix Factorization)
Decomposes the TF-IDF matrix into:
- **W** (reviews x topics): how much each review belongs to each topic
- **H** (topics x words): how much each word contributes to each topic

**Why NMF over LDA:** NMF produces more interpretable topics for short-text review data.

In [ ]:
# Build TF-IDF vectorizer with comprehensive stop words
vectorizer = TfidfVectorizer(
    max_df=0.80,        # Ignore words in >80% of reviews
    min_df=3,           # Ignore words in <3 reviews
    max_features=800,   # Keep top 800 features
    stop_words=list(CUSTOM_STOPS),
    ngram_range=(1, 2), # Single words + two-word phrases
    token_pattern=r'(?u)\b[a-zA-Z]{3,}\b'  # Alphabetic, 3+ chars only
)

texts = df['text'].tolist()
tfidf_matrix = vectorizer.fit_transform(texts)
feature_names = vectorizer.get_feature_names_out()

print(f'TF-IDF matrix: {tfidf_matrix.shape[0]} reviews x {tfidf_matrix.shape[1]} features')
print(f'\nSample features: {list(feature_names[:15])}')
print(f'Sample bigrams: {[f for f in feature_names if " " in f][:10]}')

In [ ]:
# TF-IDF for a single review — what makes it distinctive
sample_tfidf = tfidf_matrix[0].toarray().flatten()
top_indices = sample_tfidf.argsort()[-8:][::-1]

print(f'Review: {texts[0][:100]}...')
print(f'\nTop distinctive words (highest TF-IDF):')
for idx in top_indices:
    if sample_tfidf[idx] > 0:
        print(f'  "{feature_names[idx]}": {sample_tfidf[idx]:.4f}')

In [ ]:
# Run NMF topic modeling
n_topics = 8

nmf = NMF(n_components=n_topics, random_state=42, max_iter=500)
W = nmf.fit_transform(tfidf_matrix)  # Reviews -> Topics
H = nmf.components_                   # Topics -> Words

print(f'W matrix: {W.shape} (reviews x topics)')
print(f'H matrix: {H.shape} (topics x words)')

In [ ]:
# Auto-label topics based on word patterns
def label_topic(words):
    joined = ' '.join([w.lower() for w in words])
    if any(w in joined for w in ['workiva', 'vendor']): return 'Competitive landscape'
    elif any(w in joined for w in ['das', 'onpoint', 'suite']): return 'DAS & OnPoint ecosystem'
    elif 'trial balance' in joined: return 'Trial balance & tax workflows'
    elif 'customer service' in joined: return 'Customer service experience'
    elif any(w in joined for w in ['cloud', 'desktop', 'version', 'migration']): return 'Cloud vs. desktop migration'
    elif any(w in joined for w in ['data', 'import', 'extract', 'idea']): return 'Data import & analytics (IDEA)'
    elif any(w in joined for w in ['training', 'curve', 'steep']): return 'Learning curve & training'
    elif any(w in joined for w in ['engagement', 'firm', 'cch']): return 'Engagement workflows'
    elif any(w in joined for w in ['excel', 'functionality', 'feature']): return 'Features & Excel comparison'
    elif any(w in joined for w in ['audit', 'paper', 'financial']): return 'Audit & financial reporting'
    elif any(w in joined for w in ['price', 'cost', 'license', 'subscription']): return 'Pricing & licensing'
    elif any(w in joined for w in ['tax', 'returns', 'preparation']): return 'Tax preparation'
    else: return None  # Skip noise topics

topic_counts = pd.Series(W.argmax(axis=1)).value_counts().sort_index()

print('Discovered topics:')
print()
for i, topic in enumerate(H):
    top_indices = topic.argsort()[-8:][::-1]
    words = [feature_names[j] for j in top_indices]
    # Filter duplicate bigrams
    words = [w for w in words if len(w.split()) == 1 or w.split()[0] != w.split()[-1]]
    label = label_topic(words)
    count = topic_counts.get(i, 0)
    
    # Topic sentiment
    t_indices = [idx for idx, t in enumerate(W.argmax(axis=1)) if t == i]
    t_sent = df.iloc[t_indices]['vader_compound'].mean() if t_indices else 0
    
    status = 'SHOW' if label else 'SKIP'
    print(f'[{status}] {label or "Noise"} ({count} reviews, sentiment: {t_sent:.2f})')
    print(f'  Keywords: {" / ".join(words[:6])}')
    print()

---
## Step 6: Product Strengths

Same keyword approach as pain points, applied to positive reviews and pros sections.

In [ ]:
strength_categories = {
    'Comprehensive Audit Platform': ['comprehensive', 'complete', 'all in one', 'robust', 'powerful'],
    'Document Organization': ['organize', 'organization', 'structured', 'repository', 'document', 'filing'],
    'Financial Reporting': ['financial statement', 'reporting', 'financial report', 'gifi', 'statements'],
    'Collaboration': ['collaborate', 'collaboration', 'team', 'real time', 'together'],
    'Cloud Accessibility': ['cloud', 'anywhere', 'remote', 'access', 'browser', 'online'],
    'Automation & Efficiency': ['automate', 'automation', 'automatic', 'efficient', 'time saving', 'streamline'],
    'Compliance & Standards': ['compliance', 'standard', 'cas', 'aspe', 'ifrs', 'regulatory', 'methodology'],
    'Trial Balance & Working Papers': ['trial balance', 'working paper', 'lead sheet', 'engagement', 'workpaper']
}

all_positive = df[df['sentiment'] == 'Positive']['text'].tolist() + df[df['pros'].str.len() > 5]['pros'].tolist()

print('Product strengths:')
for cat, keywords in strength_categories.items():
    matches = sum(1 for t in all_positive if any(kw in t.lower() for kw in keywords))
    if matches > 2:
        print(f'  {cat}: {matches} positive mentions')

---
## Step 7: Summary — PS Team Implications

In [ ]:
print('=' * 60)
print('CASEWARE CUSTOMER VOICE — ANALYSIS SUMMARY')
print('=' * 60)

print(f'\nDataset: {len(df)} reviews from {len(df["source"].unique())} sources')
print(f'Sentiment: {len(df[df.sentiment=="Positive"])} positive, '
      f'{len(df[df.sentiment=="Neutral"])} neutral, '
      f'{len(df[df.sentiment=="Negative"])} negative')

print(f'\nTOP 3 PAIN POINTS:')
for cat, data in list(pain_results.items())[:3]:
    print(f'  {cat}: {data["count"]} mentions ({data["pct"]}%)')

print(f'\nPS TEAM IMPLICATIONS:')
if pain_results.get('User Interface & Usability', {}).get('count', 0) > 3:
    print('  - UI/UX is #1 concern. Extra training time on shortcuts and customization.')
if pain_results.get('Learning Curve & Training', {}).get('count', 0) > 3:
    print('  - Learning curve is significant. Budget extra training in SOW.')
if pain_results.get('Cloud & Migration', {}).get('count', 0) > 2:
    print('  - Cloud migration generates friction. Structured change management needed.')

reddit_mean = df[df['source'] == 'Reddit']['vader_compound'].mean()
g2_mean = df[df['source'] == 'G2']['vader_compound'].mean()
print(f'\n  - Reddit sentiment ({reddit_mean:.2f}) vs G2 ({g2_mean:.2f}):')
print('    Community forums are harsher. New users may arrive with negative')
print('    preconceptions. Proactive onboarding and early wins are essential.')